# L04 · 正弦波（sinusoid / sine wave）三要素——频率（frequency）决定音高、振幅（amplitude）决定响度、相位（phase，φ）决定起点

**学习目标**
1. 掌握 `A · sin(2πft + φ)` 的三个参数各控制什么
2. 实现 `sinusoid(t, A, f, phi)` 并用 assert 验证
3. 理解频率叠加：和弦（chord） = 多个正弦波相加
4. 为 L05（复数，complex number）和 L06（欧拉公式，Euler's formula）打好基础

**Aurora 连接**：`aurora.audio.sine` 就是这节课的核心函数；
L37 DFT 把任意信号分解成正弦波之和——分解的「原子」就是本节的 `sinusoid`。

**来自 L03 的记忆**：谱图（spectrogram）上的每条水平亮线，对应的就是一个特定频率的正弦波。
本课让你亲手控制那条线的位置（频率）、亮度（振幅）和起始角度（相位）。

## 本课剧情：操作三个旋钮

一个正弦波完全由三个参数决定：

| 参数 | 符号 | 物理意义 | 谱图上的效果 |
|---|---|---|---|
| 频率 | f（Hz）| 每秒振荡多少次 → 音高 | 水平线的 Y 轴位置 |
| 振幅 | A    | 峰值大小 → 响度 | 水平线的颜色亮度 |
| 相位 | φ（rad）| 从哪个角度开始 → 时间偏移 | （相位谱（phase spectrum），本课感受即可）|

把它们分开改，波形变化一目了然。
本课结束后，你能造任意音高、响度、起始位置的正弦波。

## 1. sin 与 cos

## 实验入口：角度、坐标和旋转

sin 和 cos 描述同一个圆周运动的两个分量：cos 比 sin 超前四分之一个周期。观察两条曲线的峰值、零点、谷值的位置关系，确认 cos(0)=1 而 sin(0)=0。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 2*np.pi, 400)
plt.plot(t, np.sin(t), label='sin')
plt.plot(t, np.cos(t), label='cos')
plt.legend(); plt.title('sin & cos'); plt.grid(True, alpha=.3); plt.show()

## 动手观察：四个特殊角度

0、π/2、π、3π/2 这四个角度的 cos 和 sin 值都是 0 或 ±1，可以直接心算验证。运行后比对 `z.real` 和 `z.imag`，确认输出与预期完全一致。

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('角度 =', np.round(angles, 3))
print('实部 cos =', np.round(z.real, 3))
print('虚部 sin =', np.round(z.imag, 3))
print('复数 z =', np.round(z, 3))


## 代码实验：遍历频率、振幅和相位

采样率（sampling rate / sample rate，sr） 16 Hz、时长 0.5 s 只产生 8 个采样点，数字少到可以逐个检查：freq 加倍时相邻样本变化幅度翻倍，amplitude 加倍时输出的 min/max 同比例放大。

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## 2. 一个正弦波 = `A · sin(2π·f·t + φ)`

- **A 振幅** = 多响（把 `[-1, 1]` 的输出拉伸到 `[-A, A]`）
- **f 频率** = 多高，Hz（每秒完整振荡次数，决定音高）
- **φ 相位** = 起始角度偏移，单位弧度（φ=π/2 时波形从峰值开始）

三个参数互相正交：A 只缩放幅度、f 只改变时间轴上的振荡速率、φ 只偏移起始角度——改任何一个不影响另外两个。这种正交设计让合成复杂波形（如和弦、MFCC filterbank 的各个分量）时可以独立调节每个成分，而不需要重新校准其他参数。

## 3. ✏️ 实现 `sinusoid(t, A, f, phi)`

**推理路线**：
1. `t` 是时间数组（秒），`f` 是每秒完整周期数（Hz）。一个完整周期对应 `2π` 弧度，所以时刻 `t` 处的角度是 `2π·f·t`。
2. 加上初相位（initial phase） `phi`（弧度），得到完整角度序列 `2π·f·t + phi`。
3. `np.sin(...)` 把角度映射到 `[-1, 1]`；乘以 `A` 把值域扩展到 `[-A, A]`。注意顺序：先算角度，再一次性乘 A，避免多次标量乘法。

**参考输入输出**：`A=2, f=1, phi=0, t=[0, 0.25, 0.5, 0.75]` → `[0, 2, 0, -2]`（一个周期的四个采样点）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `sinusoid` 前明确三件事：
- 输入：`t`（时间数组）、`A`（振幅）、`f`（频率 Hz）、`phi`（初相位，默认 0）
- 关键步骤：计算相位序列 `2π·f·t + phi`，乘以 A
- 返回：与 `t` 同形状的浮点数组，值域在 `[-A, A]`

In [ ]:
def sinusoid(t, A, f, phi=0.0):
    # ✏️ TODO: 返回正弦波
    return None  # ← 改这里


In [ ]:
import numpy as np
t = np.linspace(0, 1, 8, endpoint=False)
y = sinusoid(t, 2.0, 1.0, 0.0)
assert np.allclose(y, 2*np.sin(2*np.pi*1.0*t)), '应等于 2·sin(2πt)'
print('✅ 通过：你能造任意振幅/频率/相位的正弦波了。')

**🔗 Aurora 连接**：`aurora.audio.sine` 就是它；你将在 L33 重写它（`my_sine`）。下一课进入复数——FFT 的输出类型。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | 过零次数≈{zero_crossings}')


## 参数实验：只改一个旋钮

把 `f` 从 1 改到 4，观察相同 `t` 范围内过零点数量变为原来的 4 倍。再把 `phi` 改到 `np.pi/2`，确认波形从余弦形状（先到峰值）开始。

## 参数实验：和弦 = 多个正弦波相加

任何复杂声音都可以分解成正弦波之和（这正是 Fourier 分析的核心思想）。
反过来，把多个不同频率的 `sinusoid` 加在一起，就能合成和弦。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr

# C 大调和弦：C4 + E4 + G4
c4 = sinusoid(t, A=1.0, f=261.6)
e4 = sinusoid(t, A=0.8, f=329.6)
g4 = sinusoid(t, A=0.6, f=392.0)
chord = (c4 + e4 + g4) / 3.0

fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
for ax, sig, label in zip(axes,
        [c4, e4, g4, chord],
        ['C4 (261.6 Hz)', 'E4 (329.6 Hz)', 'G4 (392.0 Hz)', '和弦（三者叠加）']):
    ax.plot(t, sig, lw=0.8)
    ax.set_ylabel(label, fontsize=8); ax.set_ylim(-1.4, 1.4); ax.grid(alpha=0.3)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('三个正弦波 → 叠加成和弦')
plt.tight_layout(); plt.show()
print('DFT 的任务就是从「和弦」这条曲线，分解回 C4/E4/G4 三条。')

In [ ]:
# 小检查：同一个角度，同时看 cos、sin、复数
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')


## 本课收束

现在你能用 `sinusoid(t, A, f, phi)` 生成任意振幅、频率、相位的正弦波，
并用 `assert np.allclose` 验证数值是否正确。

**和弦 = 正弦波叠加**：两个不同频率的 `sinusoid` 相加，谱图上就出现两条亮线——
这正是 DFT 反问题的直觉：「给定叠加的波形，分解出每条线的位置和高度」。

**下一课 L05**：复数几何本质——实部、虚部、模、相位。
FFT 的每个输出频点（frequency bin）都是一个复数：**模 = 该频率有多响，相位 = 时间对齐信息**。
今天的 A 和 φ，就对应那个复数的模和相位。